# 00 · Data Acquisition
**Mục tiêu:** tải đúng bốn dataset từ Hugging Face, kiểm tra **từng dataset/từng split**

In [1]:
from pathlib import Path
import os, sys, json, hashlib, csv, platform
from datetime import datetime, timezone
from pprint import pprint

HERE = Path.cwd().resolve()
ROOT = HERE.parent if HERE.name.lower() == "notebooks" else HERE
if not (ROOT / "notebooks").is_dir():
    raise RuntimeError(f"sai thư mục. hiện tại: {HERE}")
RAW = ROOT / "data" / "raw"
SILVER = ROOT / "data" / "silver"
GOLD = ROOT / "data" / "gold"
REPORTS = ROOT / "reports"
for p in (RAW, REPORTS): p.mkdir(parents=True, exist_ok=True)
print("PROJECT ROOT:", ROOT)
print("PYTHON:", sys.executable)
print("LOGICAL CORES:", os.cpu_count())

PROJECT ROOT: D:\DuLieuCuaHuu\HK1_20262027\BigData\CK_BIG
PYTHON: c:\Users\Lenovo\AppData\Local\Programs\Python\Python313\python.exe
LOGICAL CORES: 16


## Thư viện và cấu hình 4 bộ dữ liệu
Số bản ghi kỳ vọng dựa trên các bản Parquet đang dùng trong kế hoạch; nếu khác phải **dừng để điều tra**, không tự sửa cho khớp.

In [2]:
SPLITS = ("train", "validation", "test")
DATASETS = {
    "cnn_dailymail": {
        "repo": "abisee/cnn_dailymail", "prefix": "3.0.0", "revision": None,
        "id": "id", "source": "article", "target": "highlights", "language": "en",
        "expected": {"train": 287113, "validation": 13368, "test": 11490},
    },
    "xsum": {
        "repo": "EdinburghNLP/xsum", "prefix": "data", "revision": None,
        "id": "id", "source": "document", "target": "summary", "language": "en",
        "expected": {"train": 204045, "validation": 11332, "test": 11334},
    },
    "vietnews": {
        "repo": "nam194/vietnews", "prefix": "data", "revision": None,
        "id": "guid", "source": "article", "target": "abstract", "language": "vi",
        "expected": {"train": 99134, "validation": 22184, "test": 22498},
    },
    "xlsum_vi": {
        "repo": "csebuetnlp/xlsum", "prefix": "vietnamese",
        "revision": "fcbaaeaec2ec660c6164ea74e8e7371b66883670",
        "id": "id", "source": "text", "target": "summary", "language": "vi",
        "expected": {"train": 32111, "validation": 4013, "test": 4013},
    },
}

def json_write(path, obj):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding="utf-8")

def csv_write(path, rows, columns=None):
    rows=list(rows)
    cols=columns or (list(rows[0]) if rows else [])
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    with path.open("w",encoding="utf-8-sig",newline="") as f:
        w=csv.DictWriter(f,fieldnames=cols,extrasaction="ignore")
        w.writeheader(); w.writerows(rows)

def file_sha256(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda:f.read(8*1024*1024),b""):h.update(block)
    return h.hexdigest()

def load_manifest(name):
    path=RAW/name/"source_manifest.json"
    if not path.exists():
        raise FileNotFoundError(f"Chưa có {path}. Hãy chạy notebook 00 trước.")
    m=json.loads(path.read_text(encoding="utf-8"))
    if m.get("dataset") != name or not m.get("valid"):
        raise ValueError(f"Manifest chưa đạt/không khớp: {path}")
    return m

def split_files(name, split):
    m=load_manifest(name)
    if split not in m["splits"]: raise ValueError(f"Thiếu split {split}: {name}")
    fs=[RAW/name/item["path"] for item in m["splits"][split]["files"]]
    if not fs or not all(p.is_file() for p in fs):
        raise FileNotFoundError(f"Thiếu Parquet {name}/{split}. Chạy notebook 00.")
    return [str(p) for p in fs]

In [3]:
import importlib.metadata as metadata
import pyarrow.parquet as pq
from huggingface_hub import HfApi, snapshot_download
for pkg in ("huggingface_hub","pyarrow","pandas"):
    print(pkg,metadata.version(pkg))
api=HfApi()
CHECK_SHA256 = True  # Luôn bật khi xác nhận dữ liệu nguồn để dùng cho nghiên cứu.
DOWNLOAD_WORKERS = 2 # Tránh nghẽn I/O/mạng Windows.

huggingface_hub 2.4.0
pyarrow 26.0.1
pandas 2.2.3


## Hàm tải và xác minh nguồn

In [7]:
def find_parquets(files, cfg):
    selected={}
    for split in SPLITS:
        if cfg["prefix"]=="vietnamese":
            paths=[f"vietnamese/xlsum-{split}.parquet"]
            paths=[p for p in paths if p in files]
        else:
            paths=sorted(p for p in files if p.startswith(cfg["prefix"]+"/")
                and Path(p).name.startswith(split+"-") and p.endswith(".parquet"))
        if not paths:
            raise FileNotFoundError(f"Không tìm thấy parquet split={split} trong repo; ví dụ: {[x for x in files if x.endswith('.parquet')][:10]}")
        selected[split]=paths
    return selected

def validate_manifest(name, manifest):
    cfg = DATASETS[name]
    if manifest.get("dataset") != name:
        return False
    for split in SPLITS:
        meta = manifest.get("splits", {}).get(split, {})
        if meta.get("rows") != cfg["expected"][split]:
            return False
        if not meta.get("valid"):
            return False
        files = meta.get("files", [])
        if not files:
            return False
        for entry in files:
            p = RAW / name / entry["path"]
            expected_size = entry.get(
                "bytes",
                entry.get("size_bytes")
            )
            if expected_size is None:
                return False
            if not p.is_file():
                return False
            if p.stat().st_size != expected_size:
                return False
            if CHECK_SHA256:
                if file_sha256(p) != entry["sha256"]:
                    return False
    return True

def download_one(name):
    cfg=DATASETS[name]
    folder=RAW/name
    folder.mkdir(parents=True,exist_ok=True)
    manifest_file=folder/"source_manifest.json"
    if manifest_file.exists():
        old=json.loads(manifest_file.read_text(encoding="utf-8"))
        if validate_manifest(name,old):
            print(f"✓ {name}: Parquet + manifest hợp lệ, không cần tải lại.")
            return old
        raise RuntimeError(f"Manifest có nhưng kiểm tra lỗi: {manifest_file}. Hãy sửa dữ liệu nguồn/đường dẫn, KHÔNG ghi đè tự động.")
    print("\n== TẢI", name, "==")
    info=api.dataset_info(cfg["repo"],revision=cfg["revision"] or "main")
    revision=info.sha
    repo_files=api.list_repo_files(repo_id=cfg["repo"],repo_type="dataset",revision=revision)
    selected=find_parquets(set(repo_files),cfg)
    paths=[p for split in SPLITS for p in selected[split]]
    print("REPO:",cfg["repo"],"| revision:",revision)
    for p in paths: print("  ",p)
    snapshot_download(repo_id=cfg["repo"],repo_type="dataset",revision=revision,
                      allow_patterns=paths,local_dir=str(folder),max_workers=DOWNLOAD_WORKERS)
    manifest={"dataset":name,"repo":cfg["repo"],"revision":revision,
              "acquired_utc":datetime.now(timezone.utc).isoformat(),
              "language":cfg["language"],"splits":{}}
    for split,relpaths in selected.items():
        total=0; entries=[]; columns=set()
        for relpath in relpaths:
            p=folder/relpath
            if not p.is_file(): raise FileNotFoundError(p)
            meta=pq.ParquetFile(str(p))
            total+=meta.metadata.num_rows
            columns.update(meta.schema_arrow.names)
            entries.append({"path":relpath,"rows":meta.metadata.num_rows,
                 "bytes":p.stat().st_size,"sha256":file_sha256(p)})
        required={cfg["id"],cfg["source"],cfg["target"]}
        valid=(total==cfg["expected"][split] and required.issubset(columns))
        manifest["splits"][split]={"rows":total,"expected_rows":cfg["expected"][split],
                  "columns":sorted(columns),"valid":valid,"files":entries}
        print(f"  {split}: {total:,} / {cfg['expected'][split]:,} - {'PASS' if valid else 'FAIL'}")
        if not valid: raise ValueError(f"{name}/{split}: dữ liệu hoặc schema không khớp, CHƯA tạo manifest đạt.")
    manifest["total_rows"]=sum(x["rows"] for x in manifest["splits"].values())
    manifest["valid"]=True
    json_write(manifest_file,manifest)
    return manifest

## 00-A · CNN/DailyMail (article → highlights)

In [5]:
cnn_manifest = download_one("cnn_dailymail")


== TẢI cnn_dailymail ==
REPO: abisee/cnn_dailymail | revision: 96df5e686bee6baa90b8bee7c28b81fa3fa6223d
   3.0.0/train-00000-of-00003.parquet
   3.0.0/train-00001-of-00003.parquet
   3.0.0/train-00002-of-00003.parquet
   3.0.0/validation-00000-of-00001.parquet
   3.0.0/test-00000-of-00001.parquet


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

  train: 287,113 / 287,113 - PASS
  validation: 13,368 / 13,368 - PASS
  test: 11,490 / 11,490 - PASS


## 00-B · XSum (document → summary)

In [6]:
xsum_manifest = download_one("xsum")


== TẢI xsum ==
REPO: EdinburghNLP/xsum | revision: 7d4d486c2f8ef850b1a11aead99b894ff3dd7da9
   data/train-00000-of-00001.parquet
   data/validation-00000-of-00001.parquet
   data/test-00000-of-00001.parquet


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

  train: 204,045 / 204,045 - PASS
  validation: 11,332 / 11,332 - PASS
  test: 11,334 / 11,334 - PASS


## 00-C · VietNews (article → abstract)

In [8]:
vietnews_manifest = download_one("vietnews")

✓ vietnews: Parquet + manifest hợp lệ, không cần tải lại.


## 00-D · XL-Sum Vietnamese (text → summary)

In [5]:
xlsum_manifest = download_one("xlsum_vi")


== TẢI xlsum_vi ==
REPO: csebuetnlp/xlsum | revision: fcbaaeaec2ec660c6164ea74e8e7371b66883670
   vietnamese/xlsum-train.parquet
   vietnamese/xlsum-validation.parquet
   vietnamese/xlsum-test.parquet


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

  train: 32,111 / 32,111 - PASS
  validation: 4,013 / 4,013 - PASS
  test: 4,013 / 4,013 - PASS


## Kiểm tra 12 split và lưu báo cáo tổng hợp

In [9]:
rows=[]
for name in DATASETS:
    m=load_manifest(name)
    for split in SPLITS:
        s=m["splits"][split]
        rows.append({"dataset":name,"split":split,"actual":s["rows"],
             "expected":s["expected_rows"],"status":"PASS" if s["valid"] else "FAIL",
             "revision":m["revision"],"files":len(s["files"])})
csv_write(REPORTS/"acquisition"/"split_inventory.csv",rows)
import pandas as pd
display(pd.DataFrame(rows))
assert len(rows)==12 and all(x["status"]=="PASS" for x in rows)
print("TỔNG:",f"{sum(x['actual'] for x in rows):,}","| TẤT CẢ 12 SPLIT PASS")

,dataset,split,actual,expected,status,revision,files
0,cnn_dailymail,train,287113,287113,PASS,96df5e686bee6baa90b8bee7c28b81fa3fa6223d,3
1,cnn_dailymail,validation,13368,13368,PASS,96df5e686bee6baa90b8bee7c28b81fa3fa6223d,1
2,cnn_dailymail,test,11490,11490,PASS,96df5e686bee6baa90b8bee7c28b81fa3fa6223d,1
3,xsum,train,204045,204045,PASS,7d4d486c2f8ef850b1a11aead99b894ff3dd7da9,1
4,xsum,validation,11332,11332,PASS,7d4d486c2f8ef850b1a11aead99b894ff3dd7da9,1
5,xsum,test,11334,11334,PASS,7d4d486c2f8ef850b1a11aead99b894ff3dd7da9,1
6,vietnews,train,99134,99134,PASS,39e2837bd57c09af5ca6f80a6483d40d904ad1ca,1
7,vietnews,validation,22184,22184,PASS,39e2837bd57c09af5ca6f80a6483d40d904ad1ca,1
8,vietnews,test,22498,22498,PASS,39e2837bd57c09af5ca6f80a6483d40d904ad1ca,1
9,xlsum_vi,train,32111,32111,PASS,fcbaaeaec2ec660c6164ea74e8e7371b66883670,1


TỔNG: 722,635 | TẤT CẢ 12 SPLIT PASS
